# Transformer-এর পরিচিতি

## এই notebook সম্পর্কে

এটি `example.py`-এর হুবহু কোড, notebook-এ বিভক্ত। README-তে বলা দুটি দাবির দুটি ছোট, কংক্রিট demo:

1. একটি ন্যূনতম single-head self-attention layer from scratch (multi-head + masking + scaling-এর
   পূর্ণ justification-সহ deep dive আসবে Phase 02-তে) — প্রতিটি position অন্য প্রতিটি position-কে দেখে।
2. একটি sequential (RNN-এর মতো) computation বনাম একটি parallel (self-attention-এর মতো)
   computation-এর কাঠামোগত তুলনা, সরাসরি সময় মেপে — যাতে 'parallelization-ই Transformers জয়ের
   কারণ' শুধু বলা নয়, কংক্রিট হয়ে ওঠে।

**চালানো:** মূল ফোল্ডারে `python example.py`, অথবা এই notebook-এর cell-গুলো ক্রমান্বয়ে চালান।

In [ ]:
import time
import numpy as np

rng = np.random.default_rng(0)

## ১. softmax (row-wise)

প্রতিটি row আলাদাভাবে softmax করা হয়, যাতে প্রতি query token-এর attention weight-গুলোর
যোগফল 1 হয়।

In [ ]:
def softmax_rows(scores):
    shifted = scores - scores.max(axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / exps.sum(axis=-1, keepdims=True)

## ২. Minimal single-head self-attention

প্রতিটি position তার Q, K, V প্রতিনিধিত্ব বের করে; প্রতিটি জোড়া token-এর মধ্যে scaled
dot-product score নিয়ে softmax করা হয়, আর সেই weight দিয়ে value-গুলোর weighted sum হয়
output। `self_attention_demo()` একটি টয় sentence-এ attention weight matrix ছাপে।

In [ ]:
def self_attention(X, Wq, Wk, Wv):
    """X: (T, d_model)। প্রতিটি position প্রতিটি position-কে দেখে, নিজেকেও —
    এটি পূর্ববর্তী পাঠের Seq2Seq attention-এর পূর্ণ generalisation, যেখানে queries,
    keys এবং values — সবই একই sequence X থেকে আসে।"""
    Q = X @ Wq   # (T, d_k)
    K = X @ Wk   # (T, d_k)
    V = X @ Wv   # (T, d_v)

    d_k = Q.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d_k)   # (T, T) -- scaling-এর আভাস, বিস্তারিত Phase 02-তে
    weights = softmax_rows(scores)       # (T, T), প্রতিটি row-এর যোগফল 1
    output = weights @ V                 # (T, d_v)
    return output, weights


def self_attention_demo():
    print("=" * 70)
    print("1. MINIMAL SELF-ATTENTION: EVERY TOKEN ATTENDS TO EVERY TOKEN")
    print("=" * 70)

    tokens = ["the", "cat", "sat", "on", "the", "mat"]
    T, d_model, d_k = len(tokens), 12, 8

    X = rng.normal(size=(T, d_model))          # প্রতিটি token-এর জন্য টয় "embedding"
    Wq = rng.normal(scale=0.3, size=(d_model, d_k))
    Wk = rng.normal(scale=0.3, size=(d_model, d_k))
    Wv = rng.normal(scale=0.3, size=(d_model, d_k))

    output, weights = self_attention(X, Wq, Wk, Wv)

    print(f"tokens = {tokens}")
    print(f"input shape {X.shape} -> output shape {output.shape}")
    print("\nAttention weight matrix (rows = query token, columns = key token,")
    print("each row sums to 1 -- read row i as 'how much token i looks at every")
    print("other token'):\n")
    header = "        " + "".join(f"{t:>7s}" for t in tokens)
    print(header)
    for i, row in enumerate(weights):
        row_str = "".join(f"{w:7.3f}" for w in row)
        print(f"{tokens[i]:>7s} {row_str}")
    print("\nNote both repeated occurrences of 'the' (positions 0 and 4) get their")
    print("own independently computed attention row -- position alone doesn't")
    print("determine the pattern, content does (this is randomly initialized and")
    print("untrained, so the specific pattern isn't meaningful yet -- Phase 02")
    print("trains this end to end).")


self_attention_demo()

## ৩. Sequential বনাম parallel computation (সময় মেপে)

`sequential_processing` একটি RNN-এর মতো ধাপে ধাপে (প্রতি ধাপ আগেরটির উপর নির্ভরশীল);
`parallel_processing` পুরো sequence-কে এক ঝটকায় matrix multiplication দিয়ে প্রক্রিয়া করে।
`timing_demo` দুটির সময় তুলনা করে — লক্ষ্য করো: single core-এ attention-এর O(T²) কাজ বেশি
সময় নেয়, কিন্তু তার T² dot product-এর মধ্যে কোনো dependency নেই, তাই GPU-তে সবগুলো
একই সময়ে গণনা করা যায়।

In [ ]:
def sequential_processing(X, W):
    """একটি RNN-এর সিমুলেশন: h_t নির্ভর করে h_{t-1}-এর উপর, ফলে একটি কঠোর
    Python loop-এ Tটি sequential ধাপ চালাতে হয় যাকে vectorize করা যায় না।"""
    T, d = X.shape
    h = np.zeros(d)
    for t in range(T):
        h = np.tanh(X[t] + h @ W)   # প্রতিটি ধাপকে অবশ্যই আগের ধাপের জন্য অপেক্ষা করতে হয়
    return h


def parallel_processing(X, Wq, Wk, Wv):
    """Self-attention-এর সিমুলেশন: গোটা sequence একসাথে আচ্ছাদন করা matrix
    multiplication-এর এক ঝটকা — কোনো ধাপকে অন্য ধাপের জন্য অপেক্ষা করতে হয় না।"""
    output, _ = self_attention(X, Wq, Wk, Wv)
    return output


def timing_demo():
    print("\n" + "=" * 70)
    print("2. SEQUENTIAL (RNN-LIKE) vs. PARALLEL (ATTENTION-LIKE) COMPUTATION")
    print("=" * 70)
    print("Structural path length between the first and last token:")
    print(f"  {'seq_len':>8}  {'RNN hops needed':>16}  {'self-attn hops needed':>22}")
    for T in [10, 50, 200, 1000]:
        print(f"  {T:>8}  {T - 1:>16}  {1:>22}")
    print("  -> In an RNN, information from token 0 passes through T-1 sequential")
    print("     steps to reach the last token. In self-attention it's always a")
    print("     single dot product, no matter how far apart the tokens are.\n")

    d = 64
    print(f"{'seq_len':>8}  {'sequential time (s)':>20}  {'parallel time (s)':>18}")
    for T in [50, 200, 800, 3000]:
        X = rng.normal(size=(T, d))
        W = rng.normal(scale=0.1, size=(d, d))
        Wq = rng.normal(scale=0.1, size=(d, d))
        Wk = rng.normal(scale=0.1, size=(d, d))
        Wv = rng.normal(scale=0.1, size=(d, d))

        start = time.perf_counter()
        sequential_processing(X, W)
        seq_time = time.perf_counter() - start

        start = time.perf_counter()
        parallel_processing(X, Wq, Wk, Wv)
        par_time = time.perf_counter() - start

        print(f"{T:>8}  {seq_time:>20.6f}  {par_time:>18.6f}")

    print("\n-> Notice the 'parallel' version actually gets SLOWER than the")
    print("   sequential one as T grows large -- and that's an honest, important")
    print("   result, not a bug. Self-attention does O(T^2) total work (a full")
    print("   TxT score matrix), while the RNN loop does only O(T) work overall.")
    print("   On a single CPU core, more total work simply takes more time.")
    print("   The point was never that attention does LESS work -- it's that")
    print("   attention's T^2 dot products have NO dependencies between them, so")
    print("   a GPU with thousands of cores can compute all of them at once,")
    print("   while the RNN's T steps have a hard dependency chain and can only")
    print("   ever run one after another, however many cores you throw at it.")
    print("   This single-threaded benchmark can't reward that property -- it")
    print("   takes real parallel hardware to see the wall-clock win -- but it")
    print("   does make the O(T) vs. O(T^2) compute trade-off from the README")
    print("   completely concrete.")


timing_demo()

## ৪. main()
`main()` দুটি demo-ই চালায় — উপরের cell-গুলোতে প্রতিটি demo তার সংজ্ঞার পরেই
ইতিমধ্যে একবার চালানো হয়েছে; `main()` এগুলোর পূর্ণ-চলমান সমতুল্য।

In [ ]:
def main():
    self_attention_demo()
    timing_demo()

In [ ]:
main()